In [5]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".keras"):
            print(os.path.join(root, file))

/kaggle/input/models/hemasakshis/dr-classification-model-/tensorflow2/default/1/final_dr_model.keras


In [7]:
import tensorflow as tf

MODEL_PATH = "/kaggle/input/models/hemasakshis/dr-classification-model-/tensorflow2/default/1/final_dr_model.keras"

cnn_model = tf.keras.models.load_model(MODEL_PATH)

print("CNN model loaded successfully!")
cnn_model.summary()

CNN model loaded successfully!


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │        10,245 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53,574,545 (204.37 MB)

 Trainable params: 14,988,293 (57.18 MB)

 Non-trainable params: 8,609,664 (32.84 MB)

 Optimizer params: 29,976,588 (114.35 MB)

In [8]:
import os

DATASET_PATH = "/kaggle/input/competitions/aptos2019-blindness-detection"

print(os.listdir(DATASET_PATH))

['sample_submission.csv', 'train_images', 'train.csv', 'test.csv', 'test_images']


In [9]:
import pandas as pd
import os

# Load the dataset labels
csv_path = os.path.join(DATASET_PATH, "train.csv")
df = pd.read_csv(csv_path)

# Select the first retinal image
sample_image_name = df.iloc[0]["id_code"]
actual_label = df.iloc[0]["diagnosis"]

# Get its image path
image_path = os.path.join(
    DATASET_PATH,
    "train_images",
    sample_image_name + ".png"
)

print("Image name:", sample_image_name)
print("Actual DR grade:", actual_label)
print("Image path:", image_path)

Image name: 000c1434d8d7
Actual DR grade: 2
Image path: /kaggle/input/competitions/aptos2019-blindness-detection/train_images/000c1434d8d7.png


In [10]:
import tensorflow as tf

# Read the retinal image
image = tf.io.read_file(image_path)

# Decode the image into RGB format
image = tf.image.decode_png(image, channels=3)

# Resize to the model's expected input size
image = tf.image.resize(image, [224, 224])

# Apply ResNet50 preprocessing
image = tf.keras.applications.resnet50.preprocess_input(image)

# Add batch dimension
image = tf.expand_dims(image, axis=0)

print("Prepared image shape:", image.shape)
print("Image preprocessing completed!")

Prepared image shape: (1, 224, 224, 3)
Image preprocessing completed!


In [11]:
# Get predictions from the CNN
predictions = cnn_model.predict(image)

# Get the class with the highest probability
predicted_class = predictions.argmax(axis=1)[0]

# Get the confidence score
confidence = predictions[0][predicted_class]

# DR class names
class_names = [
    "No DR",
    "Mild",
    "Moderate",
    "Severe",
    "Proliferative DR"
]

print("Predicted DR Grade:", class_names[predicted_class])
print("Confidence:", round(float(confidence) * 100, 2), "%")

print("\nClass-wise probabilities:")

for i, probability in enumerate(predictions[0]):
    print(class_names[i], ":", round(float(probability) * 100, 2), "%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
Predicted DR Grade: Moderate
Confidence: 69.47 %

Class-wise probabilities:
No DR : 0.02 %
Mild : 2.55 %
Moderate : 69.47 %
Severe : 4.39 %
Proliferative DR : 23.56 %


In [12]:
print("Actual DR Grade:", class_names[int(actual_label)])
print("Predicted DR Grade:", class_names[predicted_class])

if int(actual_label) == predicted_class:
    print("Result: Correct Prediction")
else:
    print("Result: Incorrect Prediction")

Actual DR Grade: Moderate
Predicted DR Grade: Moderate
Result: Correct Prediction


In [13]:
# Convert CNN prediction into structured data
cnn_output = {
    "predicted_class": class_names[predicted_class],
    "confidence": float(confidence),
    "probabilities": {
        class_names[i]: float(predictions[0][i])
        for i in range(len(class_names))
    }
}

print(cnn_output)

{'predicted_class': 'Moderate', 'confidence': 0.6947429180145264, 'probabilities': {'No DR': 0.00021847766765858978, 'Mild': 0.025516953319311142, 'Moderate': 0.6947429180145264, 'Severe': 0.04388175532221794, 'Proliferative DR': 0.23563992977142334}}


In [26]:
!pip install groq

In [28]:
from groq import Groq
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
groq_api_key = secrets.get_secret("DR_G_Report")

groq_client = Groq(api_key=groq_api_key)

print("Groq connected successfully!")

Groq connected successfully!


In [29]:
MODEL_NAME = "openai/gpt-oss-120b"

test_response = groq_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": "Write one sentence explaining diabetic retinopathy."
        }
    ],
    temperature=0.2
)

print(test_response.choices[0].message.content)

Diabetic retinopathy is a diabetes‑related eye disease in which high blood‑sugar damage to the retinal blood vessels causes leakage, swelling, and abnormal growth of new vessels, leading to progressive vision loss if untreated.


In [36]:
import numpy as np

# Get probabilities for the sample image
probs = predictions[0]

# Sort classes by probability
sorted_indices = np.argsort(probs)[::-1]

# Top prediction
top_index = sorted_indices[0]
top_class = class_names[top_index]
top_probability = float(probs[top_index])

# Second-highest prediction
second_index = sorted_indices[1]
second_class = class_names[second_index]
second_probability = float(probs[second_index])

# Probability difference
top_two_gap = top_probability - second_probability

# Normalized prediction entropy
entropy = -np.sum(probs * np.log(probs + 1e-10)) / np.log(5)

# Create structured CNN output
cnn_output = {
    "predicted_grade": int(top_index),
    "predicted_class": top_class,
    "softmax_probabilities": {
        class_names[i]: float(probs[i])
        for i in range(5)
    },
    "maximum_probability": top_probability,
    "top_two_classes": [
        {
            "class": top_class,
            "probability": top_probability
        },
        {
            "class": second_class,
            "probability": second_probability
        }
    ],
    "top_two_probability_gap": float(top_two_gap),
    "normalized_entropy": float(entropy)
}

print(cnn_output)

{'predicted_grade': 2, 'predicted_class': 'Moderate', 'softmax_probabilities': {'No DR': 0.00021847766765858978, 'Mild': 0.025516953319311142, 'Moderate': 0.6947429180145264, 'Severe': 0.04388175532221794, 'Proliferative DR': 0.23563992977142334}, 'maximum_probability': 0.6947429180145264, 'top_two_classes': [{'class': 'Moderate', 'probability': 0.6947429180145264}, {'class': 'Proliferative DR', 'probability': 0.23563992977142334}], 'top_two_probability_gap': 0.459102988243103, 'normalized_entropy': 0.5133930730698856}


In [37]:
print("cnn_output available:", "cnn_output" in globals())
print("predictions available:", "predictions" in globals())
print("class_names available:", "class_names" in globals())

cnn_output available: True
predictions available: True
class_names available: True


In [38]:
import json

common_cnn_input = json.dumps(cnn_output, indent=4)

print(common_cnn_input)

{
    "predicted_grade": 2,
    "predicted_class": "Moderate",
    "softmax_probabilities": {
        "No DR": 0.00021847766765858978,
        "Mild": 0.025516953319311142,
        "Moderate": 0.6947429180145264,
        "Severe": 0.04388175532221794,
        "Proliferative DR": 0.23563992977142334
    },
    "maximum_probability": 0.6947429180145264,
    "top_two_classes": [
        {
            "class": "Moderate",
            "probability": 0.6947429180145264
        },
        {
            "class": "Proliferative DR",
            "probability": 0.23563992977142334
        }
    ],
    "top_two_probability_gap": 0.459102988243103,
    "normalized_entropy": 0.5133930730698856
}


In [41]:
zero_shot_prompt = f"""
You are an AI-assisted medical report writing assistant
for an experimental Diabetic Retinopathy screening system.

Your task is to generate a clear, professional and
scientifically cautious screening report using ONLY
the CNN output provided below.

CNN OUTPUT:
{common_cnn_input}

Follow this report structure:

1. Screening Result
- State the predicted DR grade and category.
- Clearly identify it as a CNN-generated prediction.

2. Class-wise Probability Distribution
- Include all five DR categories.
- Convert probabilities into percentages.
- Identify the highest and second-highest predicted classes.
- Ensure the values accurately match the CNN output.

3. Prediction Uncertainty
- Explain the maximum softmax probability.
- Explain the difference between the top two class probabilities.
- Explain normalized entropy as an experimental
  indicator of prediction uncertainty.
- Do not describe these values as calibrated clinical
  confidence or actual disease probabilities.
- Do not make unsupported claims about disease severity.

4. Model Limitations
- State that this is an experimental AI model.
- Explain that it has not been clinically validated.
- Clarify that the LLM only interprets numerical CNN
  outputs and does not examine the retinal image.
- Do not invent retinal findings, patient information,
  symptoms, medical history or clinical observations.

5. Recommendation
- Recommend assessment of the original retinal image
  by a qualified ophthalmologist or eye-care professional.
- Clarify that the prediction is not a confirmed diagnosis.

Additional requirements:
- Use professional, simple medical language.
- Avoid unnecessary repetition.
- Do not recommend medication or treatment.
- Do not claim that the AI can replace a clinician.
- Do not add information that is absent from the CNN output.

Generate a complete, well-organized screening report.
"""

In [42]:
response_zero_shot = groq_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": zero_shot_prompt
        }
    ],
    temperature=0.2
)

report_zero_shot = response_zero_shot.choices[0].message.content

print(report_zero_shot)

**Diabetic Retinopathy Screening Report**  
*Generated from experimental CNN output (no image review performed)*  

---

### 1. Screening Result
- **Predicted DR grade:** **2 (Moderate)**  
- This prediction is produced by a convolutional neural network (CNN) model and reflects the model’s most likely classification for the supplied retinal image.

---

### 2. Class‑wise Probability Distribution
| DR Category | Softmax Probability | Percentage |
|-------------|--------------------|------------|
| No DR | 0.000218 | **0.02 %** |
| Mild | 0.025517 | **2.55 %** |
| **Moderate** | **0.694743** | **69.47 %** |
| Severe | 0.043882 | **4.39 %** |
| Proliferative DR | 0.235640 | **23.56 %** |

- **Highest‑probability class:** *Moderate* (≈ 69.5 %).  
- **Second‑highest‑probability class:** *Proliferative DR* (≈ 23.6 %).  

---

### 3. Prediction Uncertainty
- **Maximum softmax probability:** The model assigns a probability of **0.6947** (≈ 69.5 %) to the *Moderate* class. This is the strongest

In [43]:
with open("report_zero_shot.txt", "w", encoding="utf-8") as f:
    f.write(report_zero_shot)

print("Improved Zero-Shot report saved successfully!")

Improved Zero-Shot report saved successfully!
